# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (`data.csv`), dictionary in `columns.csv`.

Plan: inspect data → check for leakage → build features available at snapshot time → validate **out of time** → report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
print(df.shape, df.account_ref.nunique(),'accounts;', df.snapshot_month.min(),'to',df.snapshot_month.max())
print(df.isna().sum().sum(),'missing;', df.duplicated(['account_ref','snapshot_month']).sum(),'duplicate account-months')
print('churn rate', df.churn_next_month.mean().round(4))
df.describe().T

## 1. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05* — i.e. it is only known after the account's life is over, so it encodes the future. Compare it to `tenure_months` (months so far).

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('P(churn | lifetime==tenure) =', df.loc[df.gap==0,'churn_next_month'].mean())
print('P(churn | lifetime!=tenure) =', df.loc[df.gap!=0,'churn_next_month'].mean())
print('AUC of -gap alone:', roc_auc_score(df.churn_next_month, -df.gap))

Every churn row has `lifetime == tenure` (gap 0) and every non-churn row has a positive gap, so the column is the label in disguise. It would not exist at prediction time, so **it is excluded from all models**. (Shown below only to quantify how misleading it would be.)

## 2. Exploration

In [ ]:
m=df.groupby('snapshot_month').churn_next_month.agg(['mean','size'])
fig,ax=plt.subplots(1,3,figsize=(16,3.5))
m['mean'].plot(ax=ax[0],marker='o'); ax[0].set_title('Churn rate by snapshot month'); ax[0].tick_params(axis='x',rotation=90)
df.groupby('tenure_months').churn_next_month.mean().head(18).plot(ax=ax[1],marker='o'); ax[1].set_title('Churn rate by tenure')
df.groupby(pd.qcut(df.usage_hours,10,duplicates='drop'),observed=True).churn_next_month.mean().plot(ax=ax[2],marker='o'); ax[2].set_title('Churn rate by usage decile'); ax[2].tick_params(axis='x',rotation=60)
plt.tight_layout(); plt.show()
print(df.groupby('plan').churn_next_month.agg(['mean','size']))
print(df.groupby('support_tickets').churn_next_month.agg(['mean','size']))

The churn rate swings strongly with calendar month (≈3% in spring 2024 → >50% around Dec 2024–Jan 2025 → ~10% in 2025). Two consequences: (a) a random split would flatter the model, so we validate **by time**; (b) the early/late months have different cohort mixes, so performance on the most recent period is the honest estimate.

## 3. Features (only information known at the snapshot)
Own-snapshot columns plus lags from the same account's *earlier* snapshots. Calendar month is not used as a feature (the rate regime shift can't be extrapolated).

In [ ]:
d=df.drop(columns=['gap','lifetime_months']).copy()
d['month']=pd.PeriodIndex(d.snapshot_month,freq='M')
d=d.sort_values(['account_ref','month'])
g=d.groupby('account_ref')
d['usage_prev']=g.usage_hours.shift(1)
d['usage_chg']=d.usage_hours-d.usage_prev
d['tickets_prev']=g.support_tickets.shift(1)
d['usage_avg3']=g.usage_hours.transform(lambda s:s.rolling(3,min_periods=1).mean())
d['fee_per_seat']=d.monthly_fee/d.seats
d['usage_per_seat']=d.usage_hours/d.seats
d['plan_c']=d.plan.astype('category').cat.codes
FEATS=['plan_c','seats','monthly_fee','tenure_months','usage_hours','support_tickets','fee_per_seat','usage_per_seat','usage_prev','usage_chg','tickets_prev','usage_avg3']
d=d.sort_values('month').reset_index(drop=True)
# temporal split: train on the earlier ~70% of months, test on the latest
months=sorted(d.month.unique()); cut=months[int(len(months)*0.7)]
tr,te=d[d.month<cut],d[d.month>=cut]
print('train',tr.month.min(),'-',tr.month.max(),len(tr),'rows, churn',tr.churn_next_month.mean().round(3))
print('test ',te.month.min(),'-',te.month.max(),len(te),'rows, churn',te.churn_next_month.mean().round(3))

## 4. Models (out-of-time test)

In [ ]:
y_tr,y_te=tr.churn_next_month,te.churn_next_month
models={
 'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'Gradient boosting':HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0)}
res=[];P={}
for n,mdl in models.items():
    X_tr,X_te=tr[FEATS].fillna(-1),te[FEATS].fillna(-1)
    mdl.fit(X_tr,y_tr);p=mdl.predict_proba(X_te)[:,1];P[n]=p
    res.append((n,roc_auc_score(y_te,p),average_precision_score(y_te,p)))
res.append(('Baseline (base rate)',0.5,y_te.mean()))
# for comparison only: the leaky column (full data)
res.append(('LEAKY: -(lifetime-tenure)',roc_auc_score(df.churn_next_month,-df.gap),average_precision_score(df.churn_next_month,-df.gap)))
pd.DataFrame(res,columns=['model','ROC AUC','PR AUC']).round(3)

In [ ]:
# Lift: how concentrated is churn among the highest-risk accounts?
p=P['Gradient boosting']; t=pd.DataFrame({'p':p,'y':y_te.values})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
print(t.groupby('decile',observed=True).y.agg(['mean','sum']).rename(columns={'mean':'churn rate','sum':'churners'}).round(3))
print('Top-decile lift: %.2fx'%(t[t.decile==10].y.mean()/t.y.mean()))
from sklearn.inspection import permutation_importance
r=permutation_importance(models['Gradient boosting'],te[FEATS].fillna(-1),y_te,scoring='roc_auc',n_repeats=5,random_state=0)
pd.Series(r.importances_mean,FEATS).sort_values().plot.barh(title='Permutation importance (AUC drop, test)');plt.show()

In [ ]:
# Stability: AUC per test month
out=te.assign(p=P['Gradient boosting']).groupby('month').apply(lambda x:pd.Series({'n':len(x),'churn':x.churn_next_month.mean(),'AUC':roc_auc_score(x.churn_next_month,x.p) if x.churn_next_month.nunique()>1 else np.nan}),include_groups=False)
out.round(3)